# 论文图件基础复现

按顺序执行：**数据处理 → 分析 → 建模 → 绘图**。本 notebook 使用独立衍生输入包；所有项目代码均在当前 `paper_code/` 内。原始脑电全流程和论文规模模型搜索不在本 notebook 中自动启动。Fig2/5 保留候选身份。

In [ ]:
from pathlib import Path
import os, sys, json
from datetime import datetime, timezone
sys.dont_write_bytecode = True
for key in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(key, "1")
CODE = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / "paper_paths.py").is_file())
sys.path.insert(0, str(CODE))
from tools.runtime_guard import install_guard, local_import_report
install_guard()
from paper_paths import INPUT_ROOT, OUTPUT_ROOT, require_inputs, new_output
RUN = new_output("notebook_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%f"))
print({"code": str(CODE), "inputs": str(INPUT_ROOT), "outputs": str(RUN)})
print(require_inputs())

## 1. 数据处理：参与通道与事件时序

这里读取真实 Y1 衍生事件数组，执行本包复制的 rank 掩码实现，核对非参与通道与事件标签。原始 EDF 的预处理、HFO 检测与群事件构建另由 `data_processing/run.py` 提供，须传入实际记录和患者配置。

In [ ]:
from analysis.reproduce_statistics import figure1_ranks, figure2_gain, figure4_core_count
rank_check = figure1_ranks()
rank_check

## 2. 分析：空间信息增益

问题是加入空间信息是否改善 held-out 方向一致性。统计单位为患者，对照为同一患者的 Timing-only 得分；从26人的配对表重新计算增益中位数和单侧配对 Wilcoxon，并核对冻结结果。

In [ ]:
gain_check = figure2_gain()
gain_check

## 3. 建模：原生 SNN 的小规模运行

使用复制到 `modeling/` 的 LIF、空间连接和慢变量代码，运行240神经元、100 ms 的CPU检查。关闭Z/M后必须与同随机种子的基线轨迹一致；这是实现检查，不是论文规模拟合或临床状态结论。

In [ ]:
from modeling.smoke import run_smoke
model_check = run_smoke()
model_check

## 4. 从逐次评分重新计算 Fig4 C

使用原始 reducer 处理冻结的逐次训练评分，而非仅载入已画好的曲线。1–4 core各4次重复，共同前31个epoch；重算结果须与保存的完整轨迹数组及均值/样本标准差表一致。下面调用复制的原绘图函数；教学版画布不替换正式图。

In [ ]:
from plotting.reproduce import reproduce_core_count, reproduce_figures
from IPython.display import display, Image
core_case, core_check = figure4_core_count()
print(core_check)
curve_path = reproduce_core_count(core_case, RUN / "numerical")
display(Image(filename=str(curve_path), width=800))

## 5. 重建论文图版

Fig1、Fig3从冻结图层重建；Fig4恢复作者定稿导出。Fig2重新绘制其统计面板并拼版，Fig5从指定来源拼版，二者仍待作者确认。Video 2恢复冻结动画并解码核对全部帧。这里不声称重新进行了原始脑电分析或大规模SNN实验。

In [ ]:
figure_report = reproduce_figures(RUN / "figures")
print(json.dumps(figure_report, ensure_ascii=False, indent=2))

In [ ]:
for folder, stem in [("fig1", "fig1"), ("fig2_candidate", "fig2"), ("fig3", "fig3"), ("fig4", "fig4"), ("fig5_candidate", "fig5")]:
    path = RUN / "figures" / folder / "figures" / (stem + "-complete-layout.png")
    print(folder)
    display(Image(filename=str(path), width=1000))

## 6. 复现记录

记录输入、实际核对结果和运行代码位置。基础 notebook 的成功状态只覆盖上面执行的步骤；原始数据和完整建模需按各自合同另行验收。

In [ ]:
import_check = local_import_report()
report = {"rank_check": rank_check, "spatial_gain": gain_check, "model_smoke": model_check, "core_counts": core_check, "imports": import_check, "figures": figure_report}
(RUN / "notebook_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n")
print(import_check)
print("结果目录：", RUN)